In [1]:
from utils import *

### Plotting combined heatmaps for all iis

In [2]:
# Storing grid data 
folder_relaxed = 'relaxed_data'
folder_grid = 'grid_data'
os.makedirs(folder_grid, exist_ok=True)

iis = [1,2,3,4,5,8,9,10,11,12,14,15,16,18,19,20,22,25,27,30,35,40]
#iis = [10,12]
for m1,m2 in [('Mo','Mo'),('W','W'),('Mo','W')]:
    for x1 in ['S','Se']:
        A_ML = 0.5*(a_optb88[f'{m1}{x1}2']+a_optb88[f'{m2}{x1}2']) # Ang, cell constant of relaxed monolayer (used to approximate moiré cell size)
        for stacking in ['P','AP']:
            print(f'Starting calculations for system {stacking} {m1}{x1}2/{m2}{x1}2')
            for ii in iis:  
                if stacking == 'P':
                    twist_angle = twist_angles[ii]
                elif stacking =='AP':
                    twist_angle = 60-twist_angles[ii]
                                                                                                                                       
                if m1==m2 or x1 =='Se' or ii<15:
                    filename = f'{folder_relaxed}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.xyz'
                else: # Since MoS2/WS2 do not always twirl
                    filename = f'{folder_relaxed}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}_twirl.xyz'
                
                if not os.path.exists(filename):                                                                                         
                    print(f'{filename} does not exist.')
                    continue
                
                gridfile = f'{folder_grid}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz'
                
                if os.path.exists(gridfile):
                    print(f'{gridfile} already exists.')
                    continue

                #Read geometry,make and store grid
                atoms = read(filename)
                x_sc,y_sc,d_local_sc,A_REF,X_NEG,X_POS,Y_NEG,Y_POS = get_grid_data(atoms,A_ML,twist_angle)
                np.savez(gridfile,
                         x_sc=x_sc, y_sc=y_sc, d_local_sc=d_local_sc,
                         A_REF=A_REF, X_NEG=X_NEG, X_POS=X_POS, Y_NEG=Y_NEG, Y_POS=Y_POS) 
                
                ##Get higher resolution files for domain ratio calculations in case of larger twist angles
                if ii in [10,12]:
                    
                    x_sc,y_sc,d_local_sc,A_REF,X_NEG,X_POS,Y_NEG,Y_POS = get_grid_data(atoms,A_ML,
                                                                                       twist_angle,hres=True)
                    hresfile = f'{folder_grid}/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}_hres.npz'
                    np.savez(hresfile,
                             x_sc=x_sc, y_sc=y_sc, d_local_sc=d_local_sc,
                             A_REF=A_REF, X_NEG=X_NEG, X_POS=X_POS, Y_NEG=Y_NEG, Y_POS=Y_POS) 
                                        

Starting calculations for system P MoS2/MoS2
Starting calculations for system AP MoS2/MoS2
Starting calculations for system P MoSe2/MoSe2
Starting calculations for system AP MoSe2/MoSe2
Starting calculations for system P WS2/WS2
Starting calculations for system AP WS2/WS2
Starting calculations for system P WSe2/WSe2
Starting calculations for system AP WSe2/WSe2
Starting calculations for system P MoS2/WS2
Starting calculations for system AP MoS2/WS2
Starting calculations for system P MoSe2/WSe2
Starting calculations for system AP MoSe2/WSe2


In [15]:
FIXED_COLORBAR = False
colormap='YlGnBu_r'

os.makedirs('heatmaps', exist_ok=True)
os.makedirs('heatmaps/combined', exist_ok=True)

import matplotlib.gridspec as gridspec
iis = np.array([1,2,3,4,5,10,20,30])
for m1,m2 in [('Mo','Mo'),('W','W'),('Mo','W')]:
    for x1 in ['S','Se']:
        
        A_ML = 0.5*(a_optb88[f'{m1}{x1}2']+a_optb88[f'{m2}{x1}2']) # Ang, cell constant of relaxed monolayer (used to approximate moiré cell size)
        for stacking in ['P','AP']:
            print(f'Starting calculations for system {stacking} {m1}{x1}2/{m2}{x1}2')
            
            # Initialize figure
            image_file = f'heatmaps/combined/{m1}{x1}2_{m2}{x1}2_{stacking}_combined.png'
            fig = plt.figure()
            gs1 = gridspec.GridSpec(2, 4)
            gs1.update(wspace=0.025, hspace=0.05)

            for j,ii in enumerate(iis[::-1]):

  
                if stacking == 'P':
                    twist_angle = twist_angles[ii]
                elif stacking =='AP':
                    twist_angle = 60-twist_angles[ii]
                
                data = np.load(f'grid_data/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.npz')
                # Access arrays and variables
                x_sc = data['x_sc']
                y_sc = data['y_sc']
                d_local_sc = data['d_local_sc']
                A_REF = data['A_REF'].item()  
                X_NEG = data['X_NEG'].item()
                X_POS = data['X_POS'].item()
                Y_NEG = data['Y_NEG'].item()
                Y_POS = data['Y_POS'].item()
                
                if ii==30:
                    D_MIN, D_MAX = min(d_local_sc), max(d_local_sc) # Ang, interlayer distance references (min & max) to have consistent colorbar

                ## prepare plotting
                if ii in [1,2]:
                    SIZE_PT=0.004
                elif ii in [4,5]:
                    SIZE_PT=0.001
                elif ii in [3,10]:
                    SIZE_PT=0.005
                else:
                    SIZE_PT=0.0025
                
                print(f'...plot interlayer distance map {m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.png ')
                # initialize plot object
                #fig = plt.figure() # initialize figure
                ax = plt.subplot(gs1[len(iis)-1-j])

                #ax = fig.add_subplot(2,4,len(iis)-j, aspect='equal')#,adjustable='datalim') # equal aspect for axes
                # decide parameters for line plotting
                width = 0.5
                linestyle = 'w-' 
                # get size of length bar depending on A_REF
                if A_REF==20: SIZE_BAR = 10 # 1 nm bar
                elif A_REF==50: SIZE_BAR = 20 # 2 nm bar
                elif A_REF==200: SIZE_BAR = 100 # 10 nm bar
                else: SIZE_BAR = 500 # 50 nm bar
                # get twist angle with correct number of decimals
                if twist_angle>=1 and twist_angle<=59: # two decimal places
                    angle_string = r"$\theta=$"+str("{:5.2f}".format(twist_angle) )+'°' # 2 characters before decimal, 2 after
                else: angle_string = r"$\theta=$"+str("{:5.3f}".format(twist_angle) )+'°' # 2 characters before decimal, 3 after
                # define position of length bar in bottom right
                OFFSET_X = 0.30*A_REF # position from right side
                OFFSET_Y = 0.20*A_REF # position from bottom
                WIDTH_Y =  0.04*A_REF # height of vertical lines
                # define position where twist angle is shown on top left
                #LABEL_X = -X_NEG + 2.2/50*A_REF 
                LABEL_X = -X_NEG + 3.2/50*A_REF
                #LABEL_Y = Y_POS - 7.2/50*A_REF
                LABEL_Y = Y_POS - 10/50*A_REF

                ## add auxiliary elements to plot
                ax.plot([X_POS-OFFSET_X-SIZE_BAR, X_POS-OFFSET_X], [-Y_NEG+OFFSET_Y, -Y_NEG+OFFSET_Y],
                        linestyle, linewidth=width, zorder=100) # main line
                ax.plot([X_POS-OFFSET_X-SIZE_BAR, X_POS-OFFSET_X-SIZE_BAR],[-Y_NEG+OFFSET_Y+WIDTH_Y, -Y_NEG+OFFSET_Y-WIDTH_Y], 
                        linestyle, linewidth=width, zorder=100) # left border
                ax.plot([X_POS-OFFSET_X, X_POS-OFFSET_X], [-Y_NEG+OFFSET_Y+WIDTH_Y, -Y_NEG+OFFSET_Y-WIDTH_Y],
                        linestyle, linewidth=width, zorder=100) # right border

                fontsize=4

            #     ax.text(X_POS-OFFSET_X-0.5*SIZE_BAR, -Y_NEG+OFFSET_Y+0.5*WIDTH_Y, s=str(int(SIZE_BAR/10))+" nm", 
            #             horizontalalignment='center', zorder=100,fontsize=fontsize) # add label
                ax.text(X_POS-OFFSET_X-0.5*SIZE_BAR, -Y_NEG+OFFSET_Y+1.5*WIDTH_Y, s=str(int(SIZE_BAR/10))+" nm", 
                        horizontalalignment='center', zorder=100,fontsize=fontsize,c='w') # add label
                # show twist angle in top left of plot
                ax.text(LABEL_X, LABEL_Y, s=angle_string, horizontalalignment='left', 
                        bbox=dict(boxstyle="square", ec=(0., 0., 0.), fc=(1., 1., 1.)), zorder=100,fontsize=fontsize) # add label

                ## plot interlayer distance
                # if wanted: add auxiliary points to get consistent (fixed) scale of colorbar (there might be a better way, but this works nicely as well)
                if FIXED_COLORBAR:
                    x_sc.append(-1000); y_sc.append(-1000); d_local_sc.append(D_MIN) # minimum value
                    x_sc.append(-1000); y_sc.append(-1000); d_local_sc.append(D_MAX) # maximum value
                # add scatterplot of interlayer distance
                im = ax.scatter(x_sc, y_sc, c=d_local_sc, s=SIZE_PT,cmap=colormap,
                               vmin=D_MIN, vmax=D_MAX) # scatter data

                ## finalize plot

                # set limits
                ax.set_xlim(-X_NEG, X_POS) # set x limits
                ax.set_ylim(-Y_NEG, Y_POS) # set y limits
                # remove ticks for x and y axis
                ax.get_xaxis().set_ticks([]) # don't show x-axis ticks
                ax.get_yaxis().set_ticks([]) # don't show y-axis ticks

                ax.set_aspect('equal')

            wspace = 0.05
            hspace =0.01
            for i in range(4):
                lower_ax = fig.get_axes()[i]
                upper_ax = fig.get_axes()[i+4]
                lower_pos = lower_ax.get_position()
                upper_pos = upper_ax.get_position()
                gap = upper_pos.y0-lower_pos.y1
                upper_ax.set_position([upper_pos.x0,upper_pos.y0-gap+hspace,upper_pos.width,upper_pos.height])

            #plt.subplots_adjust(wspace=0.05)
            cbar_ax = fig.add_axes([0.45, 0.19, 0.45, 0.01])  # Adjust the position and size as needed
            cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal',
                                ticks=np.arange(round(D_MIN,1), D_MAX+1E-6, 0.1))
            cbar.set_label(label="d [Å]",size=fontsize+2) # label colorbar
            cbar.ax.tick_params(labelsize=fontsize) # set tick label size of colorbar
            #plt.tight_layout() # avoid problems with cut off labels
            #plt.savefig(f'heatmaps/{m1}{x1}2_{m2}{x1}2_{stacking}_{ii}.png',dpi=300)
            #plt.savefig(f'combined_trial.png',dpi=300)
            cbar.ax.xaxis.set_label_coords(-0.05,0.7)
            plt.savefig(image_file,dpi=300,facecolor='white',transparent=False,
                        bbox_inches='tight',pad_inches=0.02)
            plt.close()


Starting calculations for system P MoS2/MoS2
...plot interlayer distance map MoS2_MoS2_P_30.png 
...plot interlayer distance map MoS2_MoS2_P_20.png 
...plot interlayer distance map MoS2_MoS2_P_10.png 
...plot interlayer distance map MoS2_MoS2_P_5.png 
...plot interlayer distance map MoS2_MoS2_P_4.png 
...plot interlayer distance map MoS2_MoS2_P_3.png 
...plot interlayer distance map MoS2_MoS2_P_2.png 
...plot interlayer distance map MoS2_MoS2_P_1.png 
Starting calculations for system AP MoS2/MoS2
...plot interlayer distance map MoS2_MoS2_AP_30.png 
...plot interlayer distance map MoS2_MoS2_AP_20.png 
...plot interlayer distance map MoS2_MoS2_AP_10.png 
...plot interlayer distance map MoS2_MoS2_AP_5.png 
...plot interlayer distance map MoS2_MoS2_AP_4.png 
...plot interlayer distance map MoS2_MoS2_AP_3.png 
...plot interlayer distance map MoS2_MoS2_AP_2.png 
...plot interlayer distance map MoS2_MoS2_AP_1.png 
Starting calculations for system P MoSe2/MoSe2
...plot interlayer distance map 

# Convert twirled MoSe2/WSe2 to MoS2/WS2

In [11]:
m1 = 'Mo'
m2 = 'W'
iis = [40] ## Specify all iis

a_scale = (a_optb88['MoS2']+a_optb88['WS2'])/(a_optb88['MoSe2']+a_optb88['WSe2'])
t_scale = (t['MoS2']+t['WS2'])/(t['MoSe2']+t['WSe2'])
d_scale = 6.21323/6.59509
a_scale,t_scale,d_scale

for stacking in ['P']:
    for ii in iis:
        filename = f'relaxed_data/{m1}Se2_{m2}Se2_{stacking}_{ii}.xyz'
        atoms = read(filename)

        for i in range(len(atoms)):
            if atoms.symbols[i]=='Se':
                atoms.symbols[i]='S'

        cell_scale = np.array([a_scale,a_scale,d_scale])

        print(atoms.cell)
        #atoms.set_cell
        atoms.set_cell(atoms.cell*cell_scale,scale_atoms=True)
        print(atoms.cell)
        from ase.io.lammpsdata import write_lammps_data
        write_lammps_data(f'rigid_data/data_{m1}S2_{m2}S2_{stacking}_{ii}_twirl',atoms)

Cell([[233.08034020371693, 0.0, 0.0], [116.54017010185845, 201.85349573913825, 0.0], [0.0, 0.0, 30.0]])
Cell([[223.68004752594612, 0.0, 0.0], [111.84002376297305, 193.7126034771799, 0.0], [0.0, 0.0, 28.26298049003122]])
